# 📺 subpas — Google Colab Notebook

> Lấy toàn bộ subscription/playlist YouTube và phân loại bằng **Gemini AI**.

**Repository gốc:** https://github.com/tpc-pascal/subpas  
**Tác giả:** tpc-pascal

---
### 🚀 Hướng dẫn nhanh
1. Chạy từng cell theo thứ tự từ trên xuống dưới.
2. Upload **Gemini API Key** (file `.txt`) và **Google OAuth Client Secret** (file `.json`).
3. Xác thực tài khoản Google → lấy mã code → dán vào ô nhập.
4. Chờ hệ thống thu thập & phân loại subscriptions.
5. Xem kết quả, duyệt playlists, tải file ZIP.

> 💡 Bạn có thể mount Google Drive để lưu dữ liệu xuất ra đó.

In [ ]:
# === CELL 1: Cài đặt dependencies ===
print("🔄 Đang cài đặt thư viện...")

!pip install -q google-api-python-client google-auth-oauthlib pandas google-genai ipywidgets

print("✅ Hoàn tất cài đặt!")

In [ ]:
# === CELL 2: Import thư viện ===
import json
import time
import re
import os
import zipfile
import warnings
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
from IPython.display import display, HTML, Image as IPyImage, clear_output

import pandas as pd
from google import genai
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 200)

print("✅ Import thành công!")

---
### 💾 (Tuỳ chọn) Mount Google Drive
Nếu muốn lưu file xuất vào Google Drive thay vì tải xuống máy, hãy chạy cell bên dưới.

In [ ]:
# === CELL 3: Mount Google Drive (tuỳ chọn) ===
MOUNT_DRIVE = False  # Đổi thành True nếu muốn mount Drive

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    EXPORT_DIR = '/content/drive/MyDrive/subpas_Export'
    os.makedirs(EXPORT_DIR, exist_ok=True)
    print(f"📁 Thư mục xuất: {EXPORT_DIR}")
else:
    EXPORT_DIR = '/content'
    print("ℹ️ Dùng thư mục mặc định (tải file xuống máy sau khi xuất).")

---
### 🔑 Bước 1: Upload credentials

| File | Nguồn |
|---|---|
| `api_key.txt` | [Google AI Studio](https://aistudio.google.com/apikey) |
| `client_secret.json` | [Google Cloud Console](https://console.cloud.google.com/apis/credentials) |

> ⚠️ Nếu không upload Gemini API Key, subscriptions sẽ không được phân loại (hiển thị `---`).

In [ ]:
# === CELL 4: Upload Gemini API Key ===
from google.colab import files

print("📤 Upload file API Key (.txt) — nhấn 'Choose File' rồi chọn file...")
uploaded_key = files.upload()

GEMINI_API_KEY = None
key_filename = None

for fname in uploaded_key:
    key_content = uploaded_key[fname].decode('utf-8').strip()
    if key_content:
        GEMINI_API_KEY = key_content
        key_filename = fname
        print(f"✅ Đã nhận API Key từ file: {fname}")
        break

if not GEMINI_API_KEY:
    print("ℹ️ Không có API Key → chế độ không phân loại.")

In [ ]:
# === CELL 5: Upload YouTube OAuth Client Secret ===
print("📤 Upload file Client Secret (.json)...")
uploaded_oauth = files.upload()

OAUTH_CONFIG = None
oauth_filename = None

for fname in uploaded_oauth:
    try:
        OAUTH_CONFIG = json.loads(uploaded_oauth[fname].decode('utf-8'))
        oauth_filename = fname
        print(f"✅ Đã nhận OAuth config từ file: {fname}")
        break
    except json.JSONDecodeError:
        print(f"❌ File {fname} không phải JSON hợp lệ!")

if not OAUTH_CONFIG:
    raise RuntimeError("❌ Cần file OAuth Client Secret (.json) để tiếp tục!")

---
### 🔧 Bước 2: Khởi tạo Gemini Classifier

In [ ]:
# === CELL 6: Khởi tạo Gemini Classifier ===
class GeminiClassifier:
    def __init__(self, api_key=None):
        self.client = None
        self.available_models = []
        self.current_model_index = 0
        if api_key:
            self._init_client(api_key)

    def _init_client(self, api_key):
        try:
            self.client = genai.Client(api_key=api_key)
            raw_models = self.client.models.list()
            for m in raw_models:
                name_lower = m.name.lower()
                if 'gemini' in name_lower:
                    supported = getattr(m, 'supported_generation_methods', []) or getattr(m, 'supported_methods', [])
                    if 'generateContent' in supported or any(x in name_lower for x in ['flash', 'pro']):
                        self.available_models.append(m.name.split('/')[-1])
            self.available_models.sort(key=lambda x: ('flash' not in x.lower(), x))
            print(f"✅ Gemini sẵn sàng. Models: {', '.join(self.available_models[:3])}...")
        except Exception as e:
            print(f"❌ Lỗi khởi tạo Gemini: {e}")

    def classify_batch(self, channel_names):
        if not channel_names or self.client is None or not self.available_models:
            return ['---'] * len(channel_names)

        prompt = (
            f"Classify {len(channel_names)} YouTube channels into these categories only: "
            "Code, Entertainment, News, AI/ML, Education, Music, Game, Tech, Other. "
            "\nOutput: CSV format only, no explanation, no channel names.\n"
            "Channels: " + ', '.join(channel_names)
        )

        attempts = 0
        max_attempts = len(self.available_models)

        while attempts < max_attempts:
            target_model = self.available_models[self.current_model_index]
            try:
                response = self.client.models.generate_content(
                    model=target_model,
                    contents=prompt,
                    config={'temperature': 0.0, 'max_output_tokens': 800}
                )
                if not response or not response.text:
                    return ['Other'] * len(channel_names)

                clean_text = response.text.strip().replace('`', '').replace('python', '').strip()
                tags = [t.strip() for t in re.split(r'[,\n;]', clean_text) if t.strip()]

                if len(tags) != len(channel_names):
                    if len(tags) < len(channel_names):
                        tags.extend(['Other'] * (len(channel_names) - len(tags)))
                    else:
                        tags = tags[:len(channel_names)]
                return tags

            except Exception as e:
                if '429' in str(e):
                    print(f"  ⚠️ Rate limit tại {target_model}, chuyển model...")
                else:
                    print(f"  ⚠️ Lỗi {target_model}: {str(e)[:60]}")
                self.current_model_index = (self.current_model_index + 1) % len(self.available_models)
                attempts += 1
                time.sleep(2)

        return ['Other'] * len(channel_names)

    def classify_parallel(self, all_channel_names, batch_size=50):
        if not all_channel_names:
            return []
        batches = [all_channel_names[i:i + batch_size] for i in range(0, len(all_channel_names), batch_size)]
        print(f"  📊 Tổng: {len(all_channel_names)} kênh, {len(batches)} lô (mỗi lô {batch_size})")

        final_results = []
        for i, batch in enumerate(batches):
            print(f"  🔄 Lô {i+1}/{len(batches)}...", end=' ')
            res = self.classify_batch(batch)
            final_results.extend(res)
            print(f"✅")
            if i < len(batches) - 1:
                time.sleep(6)
        print("  ✅ Phân loại hoàn tất!")
        return final_results

classifier = GeminiClassifier(GEMINI_API_KEY)
if not GEMINI_API_KEY:
    print("ℹ️ Chạy ở chế độ không phân loại (thiếu Gemini API Key).")

---
### 🔐 Bước 3: Xác thực YouTube

In [ ]:
# === CELL 7: Xác thực OAuth 2.0 ===
SCOPES = [
    'https://www.googleapis.com/auth/youtube.readonly',
    'https://www.googleapis.com/auth/youtube.force-ssl'
]

print("🔗 Đang tạo link xác thực...")

flow = InstalledAppFlow.from_client_config(
    OAUTH_CONFIG, SCOPES, redirect_uri='urn:ietf:wg:oauth:2.0:oob'
)
url, _ = flow.authorization_url(prompt='consent')

display(HTML(
    f'<div style="padding:16px;border:2px solid #2563eb;border-radius:12px;text-align:center;margin:12px 0;background:#f0f7ff;">'
    f'<p style="font-size:16px;margin-bottom:12px;">👉 Click nút bên dưới để xác thực Google:</p>'
    f'<a href="{url}" target="_blank" style="display:inline-block;background:#2563eb;color:white;padding:12px 32px;border-radius:8px;text-decoration:none;font-weight:bold;font-size:16px;">'
    f'🔗 Mở trang xác thực Google</a>'
    f'</div>'
))

print("\n📝 Sau khi cấp quyền, bạn sẽ nhận được một mã code. Dán mã đó vào ô bên dưới.")

In [ ]:
# === CELL 8: Dán mã xác thực ===
from getpass import getpass

auth_code = getpass("🔑 Dán mã xác thực tại đây: ").strip()

if not auth_code:
    raise ValueError("❌ Bạn chưa nhập mã xác thực!")

print("🔄 Đang trao đổi mã lấy token...")
flow.fetch_token(code=auth_code)
credentials = flow.credentials
print("✅ Xác thực thành công!")

---
### 📥 Bước 4: Thu thập Subscriptions & Phân loại AI

Quá trình này có thể mất vài phút tuỳ vào số lượng kênh bạn đăng ký.

In [ ]:
# === CELL 9: Fetch subscriptions + classify ===
print("🔧 Đang khởi tạo YouTube service...")
service = build('youtube', 'v3', credentials=credentials)

print("📥 Đang thu thập danh sách subscriptions...")
raw_items = []
next_token = None

while True:
    res = service.subscriptions().list(
        part='snippet', mine=True, maxResults=50, pageToken=next_token
    ).execute()
    raw_items.extend(res.get('items', []))
    next_token = res.get('nextPageToken')
    if not next_token:
        break
    print(f"  📄 Đã lấy {len(raw_items)} kênh...")

print(f"✅ Tổng số subscriptions: {len(raw_items)}")

channel_names = [item['snippet']['title'] for item in raw_items]

print("🤖 Đang phân loại kênh bằng Gemini AI...")
if classifier.client and classifier.available_models:
    all_tags = classifier.classify_parallel(channel_names, batch_size=50)
else:
    all_tags = ['---'] * len(channel_names)
    print("  ℹ️ Bỏ qua phân loại (không có API Key).")

all_subs = []
for idx, item in enumerate(raw_items):
    all_subs.append({
        'Tên Kênh': item['snippet']['title'],
        'Chủ đề': all_tags[idx] if idx < len(all_tags) else 'Other',
        'Hình Ảnh': item['snippet']['thumbnails'].get('high', {}).get('url'),
        'Ngày Đăng Ký': item['snippet']['publishedAt'].split('T')[0],
        'Liên Kết': f"https://www.youtube.com/channel/{item['snippet']['resourceId']['channelId']}",
        'ID': item['id']
    })

df_subs = pd.DataFrame(all_subs)
if not df_subs.empty:
    df_subs = df_subs.sort_values(by=['Chủ đề', 'Tên Kênh']).reset_index(drop=True)
    df_subs.insert(0, 'STT', range(1, len(df_subs) + 1))

print(f"✅ Hoàn tất! {len(df_subs)} subscriptions đã sẵn sàng.")

---
### 👥 Kết quả Subscriptions

In [ ]:
# === CELL 10: Hiển thị Subscriptions ===
display_df = df_subs[['STT', 'Chủ đề', 'Tên Kênh', 'Ngày Đăng Ký', 'Liên Kết']].copy()
display_df['Liên Kết'] = display_df['Liên Kết'].apply(
    lambda x: f'<a href="{x}" target="_blank">🔗 Link</a>'
)

display(HTML(
    f'<div style="margin:8px 0;">📊 <b>Tổng cộng: {len(df_subs)} subscriptions</b></div>'
))
display(HTML(display_df.to_html(escape=False, index=False)))

# Thống kê theo chủ đề
if 'Chủ đề' in df_subs.columns and classifier.client:
    print("\n📊 Thống kê theo chủ đề:")
    topic_counts = df_subs['Chủ đề'].value_counts()
    for topic, count in topic_counts.items():
        bar = '█' * count
        print(f"  {topic:15s} | {bar} {count}")

In [ ]:
# === CELL 11: Xem thumbnail kênh (tuỳ chọn) ===
# Nhập STT của kênh muốn xem thumbnail
try:
    stt = int(input("🔍 Nhập STT kênh muốn xem thumbnail (Enter để bỏ qua): ").strip())
    if 1 <= stt <= len(df_subs):
        row = df_subs.iloc[stt - 1]
        print(f"📺 {row['Tên Kênh']} — {row['Chủ đề']}")
        display(IPyImage(url=row['Hình Ảnh']))
    else:
        print("❌ STT không hợp lệ.")
except (ValueError, IndexError):
    print("ℹ️ Bỏ qua.")

---
### 📂 Bước 5: Playlists

In [ ]:
# === CELL 12: Fetch playlists ===
print("📂 Đang tải danh sách Playlists...")
res_p = service.playlists().list(
    part='snippet,contentDetails', mine=True, maxResults=50
).execute()

playlists = []
for pl in res_p.get('items', []):
    playlists.append({
        'Tên Playlist': pl['snippet']['title'],
        'Số Video': pl['contentDetails']['itemCount'],
        'ID Playlist': pl['id'],
        'Mô tả': (pl['snippet']['description'] or '')[:100]
    })

df_playlists = pd.DataFrame(playlists)
print(f"✅ Tổng số Playlists: {len(df_playlists)}")
display(HTML(df_playlists.to_html(index=False)))

In [ ]:
# === CELL 13: Xem video trong Playlist (tuỳ chọn) ===
# Nhập ID Playlist (cột bên trái) để xem video
playlist_id = input("🎬 Nhập ID Playlist muốn xem (Enter để bỏ qua): ").strip()

if playlist_id:
    print(f"📥 Đang tải video từ playlist: {playlist_id}")
    videos = []
    next_token = None
    while True:
        res = service.playlistItems().list(
            part='snippet', playlistId=playlist_id, maxResults=50, pageToken=next_token
        ).execute()
        for item in res.get('items', []):
            videos.append({
                'Tên Video': item['snippet']['title'],
                'Ngày Thêm': item['snippet']['publishedAt'].split('T')[0],
                'Thumbnail': item['snippet']['thumbnails'].get('high', {}).get('url'),
                'Link': f"https://www.youtube.com/watch?v={item['snippet']['resourceId']['videoId']}"
            })
        next_token = res.get('nextPageToken')
        if not next_token:
            break

    df_videos = pd.DataFrame(videos)
    print(f"✅ {len(df_videos)} videos")

    if not df_videos.empty:
        display_df_v = df_videos[['Tên Video', 'Ngày Thêm', 'Link']].copy()
        display_df_v['Link'] = display_df_v['Link'].apply(
            lambda x: f'<a href="{x}" target="_blank">▶️ Xem</a>'
        )
        display(HTML(display_df_v.to_html(escape=False, index=False)))

        # Cho phép xem thumbnail video
        try:
            v_idx = int(input("🔍 Nhập số thứ tự video muốn xem thumbnail (Enter để bỏ qua): ").strip())
            if 1 <= v_idx <= len(df_videos):
                vrow = df_videos.iloc[v_idx - 1]
                print(f"📺 {vrow['Tên Video']}")
                display(IPyImage(url=vrow['Thumbnail']))
        except (ValueError, IndexError):
            pass
else:
    print("ℹ️ Bỏ qua.")

---
### 📦 Bước 6: Xuất dữ liệu

Chạy cell bên dưới để xuất danh sách subscriptions ra file ZIP (kèm CSV) và tải về máy.

In [ ]:
# === CELL 14: Xuất dữ liệu ZIP ===
if df_subs is not None and len(df_subs) > 0:
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    csv_filename = f'{EXPORT_DIR}/youtube_subs_{ts}.csv'
    zip_filename = f'{EXPORT_DIR}/subpas_{ts}.zip'

    # Export CSV
    df_subs.to_csv(csv_filename, index=False, encoding='utf-8-sig')

    # Nén ZIP
    with zipfile.ZipFile(zip_filename, 'w') as zipf:
        zipf.write(csv_filename, arcname=os.path.basename(csv_filename))

    os.remove(csv_filename)

    print(f"✅ Đã xuất: {zip_filename}")

    # Tải xuống máy (nếu không mount Drive)
    if not MOUNT_DRIVE:
        from google.colab import files as colab_files
        colab_files.download(zip_filename)
        print("📥 Đã bắt đầu tải file về máy của bạn!")
else:
    print("❌ Không có dữ liệu để xuất.")

---
### ✅ Hoàn tất!

Cảm ơn bạn đã sử dụng **subpas**!  
Nếu thấy hữu ích, hãy ghé thăm [GitHub Repository](https://github.com/tpc-pascal/subpas) và để lại ⭐.

> ⚠️ **Lưu ý:** Token xác thực và API Key chỉ tồn tại trong phiên Colab này. Đóng runtime sẽ xoá toàn bộ dữ liệu.